# Camera Input for Motion Games: A Temporal Benchmark of Hand/Body Tracking

**Goal:** Pick the perception backend for *Play With Air* games — 1–4 players at 1–3 m from a phone/webcam, whole hand as pointer, fast slashing motion. Secondary goal: publish as blog post + open benchmark repo.

**The gap this fills:** Existing benchmarks (COCO mAP) measure *spatial accuracy on static images*. Games care about **temporal properties**: latency tails, dropout under fast motion, identity stability, jitter at rest. Nobody measures those. We do.

## Hypotheses (pre-registered)

- **H1:** Body-pose tracking has *lower dropout under fast hand motion* than hand-specific detectors — a slashing hand motion-blurs into an unrecognizable smear; the torso it's attached to barely moves.
- **H2:** Detection-every-frame (YOLO) has *better identity stability* than tracking-based pipelines (MediaPipe) when players' hands cross.
- **H3:** Hand detectors *degrade with distance* faster than body pose — at 3 m a hand is ~30 px, a person is ~300 px.
- **H4:** 21-keypoint hand precision is *unnecessary* for whole-hand-pointer gameplay (Nex Playground ships 18-point body tracking for the same games).

## Contenders

| # | Model | Type | Wrist source | Multi-player |
|---|-------|------|--------------|-------------|
| 1 | **YOLO26n-pose body** (pretrained COCO) | person + 17 kpts | keypoints 9, 10 | native (person detection) |
| 2 | **YOLO26n-pose hand** (ours, mAP50 90.3%) | hand + 21 kpts | keypoint 0 | needs matcher |
| 3 | **MediaPipe Hands** (current Godot backend) | hand + 21 landmarks | landmark 0 | needs matcher, tracking-based |
| 4 | **RTMPose body ONNX** (optional, via Docker export) | person + 17 kpts | keypoints 9, 10 | needs person detector |

**Common evaluation point:** every model reports a *wrist*. All metrics evaluate the same semantic point.

**Methodology docs:** [gameplay-requirements.md](../docs/research/gameplay-requirements.md) · [nex-playground-analysis.md](../docs/research/nex-playground-analysis.md) · [SOTA survey](../docs/research/edge-hand-tracking-sota-2026.md)

**Hardware (round 1):** RTX 3070 Ti Laptop GPU / i9-12900H. Mobile hardware is future work.

## Decision rule

**Hard gates** (fail any → disqualified):
- **G1** P95 latency ≤ 33 ms (30 fps frame budget — [Ultralytics real-time guidance](https://www.ultralytics.com/glossary/real-time-inference#the-importance-of-low-latency))
- **G2** Slash dropout ≤ 5% at 2 m
- **G3** Detects 2 players simultaneously at 2 m
- **G4** Viable Godot + mobile deployment path

**Then lexicographic ranking:** slash dropout @2m → P95 latency → track churn (identity) → jitter → 3 m dropout.

## 0. Setup

In [1]:
import json, time, statistics, math
from pathlib import Path
import cv2
import numpy as np

ROOT = Path('..').resolve()
CORPUS = ROOT / 'corpus'
RESULTS = ROOT / 'benchmark_results'
RESULTS.mkdir(exist_ok=True)

YOLO_BODY_PT = 'yolo26n-pose.pt'                                    # auto-downloads (COCO body)
YOLO_HAND_PT = str(ROOT / 'runs/pose/hand-yolo26n/weights/best.pt') # our trained model
RTMPOSE_ONNX = str(ROOT / 'models/rtmpose/rtmpose-s-body.onnx')     # optional (Docker export)

# Clip metadata: expected people per clip (protocol-guaranteed presence)
CLIP_PEOPLE = {
    'idle_1m': 1, 'idle_2m': 1, 'idle_3m': 1,
    'slash_1m': 1, 'slash_2m': 1, 'slash_3m': 1,
    'two_idle_2m': 2, 'two_slash_2m': 2, 'two_cross_2m': 2,
    'walkon_2m': None,  # presence varies — excluded from dropout, used for acquisition
}

available = sorted(p.stem for p in CORPUS.glob('*.mp4')) if CORPUS.exists() else []
print(f'Corpus clips available: {available or "NONE — run tools/record_corpus.py first"}')
print(f'Hand model exists: {Path(YOLO_HAND_PT).exists()}')

Corpus clips available: ['idle_1m', 'idle_2m', 'idle_3m', 'slash_1m', 'slash_2m', 'slash_3m', 'two_cross_2m', 'two_idle_2m', 'two_slash_2m', 'walkon_2m']
Hand model exists: True


## 1. Model Adapters

Uniform interface: `adapter.detect(frame_bgr) -> list[entity]` where each entity is
`{'cx','cy','wrists':[[x,y],...],'conf'}` (normalized 0..1 coords).
For body models an entity = a **person** (up to 2 wrists). For hand models an entity = a **hand** (1 wrist).
`expected_entities(people)` maps protocol people-count to expected detections.

In [2]:
class YOLOBodyAdapter:
    name, kind = 'yolo26_body', 'person'
    def __init__(self):
        from ultralytics import YOLO
        self.model = YOLO(YOLO_BODY_PT)
    def detect(self, frame):
        h, w = frame.shape[:2]
        res = self.model(frame, imgsz=640, conf=0.4, verbose=False)[0]
        out = []
        if res.keypoints is None or res.boxes is None:
            return out
        kps = res.keypoints.xy.cpu().numpy()      # (N,17,2) pixels
        boxes = res.boxes.xywh.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        for i in range(len(boxes)):
            wrists = []
            for k in (9, 10):  # left/right wrist
                x, y = kps[i][k]
                if x > 0 or y > 0:  # (0,0) = not visible
                    wrists.append([float(x/w), float(y/h)])
            out.append({'cx': float(boxes[i][0]/w), 'cy': float(boxes[i][1]/h),
                        'wrists': wrists, 'conf': float(confs[i])})
        return out
    def expected_entities(self, people): return people

class YOLOHandAdapter:
    name, kind = 'yolo26_hand', 'hand'
    def __init__(self):
        from ultralytics import YOLO
        self.model = YOLO(YOLO_HAND_PT)
    def detect(self, frame):
        h, w = frame.shape[:2]
        res = self.model(frame, imgsz=640, conf=0.4, verbose=False)[0]
        out = []
        if res.keypoints is None or res.boxes is None:
            return out
        kps = res.keypoints.xy.cpu().numpy()      # (N,21,2)
        boxes = res.boxes.xywh.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        for i in range(len(boxes)):
            x, y = kps[i][0]  # wrist = keypoint 0
            out.append({'cx': float(boxes[i][0]/w), 'cy': float(boxes[i][1]/h),
                        'wrists': [[float(x/w), float(y/h)]], 'conf': float(confs[i])})
        return out
    def expected_entities(self, people): return people * 2  # protocol: both hands up

class MediaPipeAdapter:
    """MediaPipe Tasks HandLandmarker in VIDEO mode — the SAME API our Godot
    GDMP backend uses, so this is representative of the shipping pipeline.
    VIDEO mode keeps tracking state between frames (like LIVE_STREAM in-game)."""
    name, kind = 'mediapipe_hands', 'hand'
    MODEL = str(ROOT / 'models/mediapipe/hand_landmarker.task')
    def __init__(self, max_hands=4):
        self.max_hands = max_hands
        self._ts = 0
        self._make()
    def _make(self):
        import mediapipe as mp
        from mediapipe.tasks.python import vision, BaseOptions
        opts = vision.HandLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=self.MODEL),
            running_mode=vision.RunningMode.VIDEO,
            num_hands=self.max_hands,
            min_hand_detection_confidence=0.4,
            min_hand_presence_confidence=0.4,
            min_tracking_confidence=0.3)
        self.landmarker = vision.HandLandmarker.create_from_options(opts)
        self.mp = mp
    def reset(self):
        self.landmarker.close(); self._ts = 0; self._make()  # fresh tracker per clip
    def detect(self, frame):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = self.mp.Image(image_format=self.mp.ImageFormat.SRGB, data=rgb)
        self._ts += 33  # ~30fps timestamps required by VIDEO mode
        res = self.landmarker.detect_for_video(img, self._ts)
        out = []
        for lm in res.hand_landmarks:
            xs = [p.x for p in lm]; ys = [p.y for p in lm]
            out.append({'cx': float(np.mean(xs)), 'cy': float(np.mean(ys)),
                        'wrists': [[float(lm[0].x), float(lm[0].y)]],
                        'conf': 1.0})
        return out
    def expected_entities(self, people): return people * 2
    def close(self): self.landmarker.close()

print('Adapters defined.')

Adapters defined.


### 1b. RTMPose body adapter (optional — top-down: YOLO26 person boxes → RTMPose ONNX crops)

Pre-exported ONNX from OpenMMLab (`models/rtmpose/rtmpose-s-body.onnx`, SimCC head, 256×192 input).
Latency reported = detector + pose (the full pipeline a game would pay for).

In [3]:
class RTMPoseBodyAdapter:
    name, kind = 'rtmpose_body', 'person'
    MEAN = np.array([123.675, 116.28, 103.53]); STD = np.array([58.395, 57.12, 57.375])
    def __init__(self):
        import onnxruntime as ort
        from ultralytics import YOLO
        self.det = YOLO(YOLO_BODY_PT)  # person detector (boxes only)
        self.sess = ort.InferenceSession(RTMPOSE_ONNX, providers=ort.get_available_providers())
        self.inp = self.sess.get_inputs()[0].name
    def _pose(self, crop):
        img = cv2.resize(crop, (192, 256)).astype(np.float32)
        img = (img[:, :, ::-1] - self.MEAN) / self.STD  # BGR->RGB, normalize
        blob = np.transpose(img, (2, 0, 1))[np.newaxis].astype(np.float32)
        sx, sy = self.sess.run(None, {self.inp: blob})
        # SimCC: argmax over x/y distributions, /2 (simcc_split_ratio)
        kx = sx[0].argmax(axis=1) / 2.0  # (17,) in 192-space
        ky = sy[0].argmax(axis=1) / 2.0  # (17,) in 256-space
        return kx / 192.0, ky / 256.0    # normalized within crop
    def detect(self, frame):
        h, w = frame.shape[:2]
        res = self.det(frame, imgsz=640, conf=0.4, classes=[0], verbose=False)[0]
        out = []
        if res.boxes is None: return out
        for box in res.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            x1, y1 = max(0, int(x1)), max(0, int(y1))
            x2, y2 = min(w, int(x2)), min(h, int(y2))
            if x2 - x1 < 10 or y2 - y1 < 10: continue
            kx, ky = self._pose(frame[y1:y2, x1:x2])
            wrists = []
            for k in (9, 10):
                wx = (x1 + kx[k] * (x2 - x1)) / w
                wy = (y1 + ky[k] * (y2 - y1)) / h
                wrists.append([float(wx), float(wy)])
            out.append({'cx': float((x1+x2)/2/w), 'cy': float((y1+y2)/2/h),
                        'wrists': wrists, 'conf': float(box.conf[0])})
        return out
    def expected_entities(self, people): return people

# Included automatically in the run-all cell when the ONNX file exists.
print('RTMPose adapter defined. ONNX present:', Path(RTMPOSE_ONNX).exists())

RTMPose adapter defined. ONNX present: True


## 2. Runner — inference decoupled from analysis

Each (model × clip) run dumps raw per-frame JSON to `benchmark_results/<model>/<clip>.json`.
Analysis cells only read JSON — re-analyzing never reruns models.

In [4]:
def run_model_on_clip(adapter, clip_name, warmup=10):
    if hasattr(adapter, 'reset'):
        adapter.reset()  # fresh tracker state per clip (MediaPipe)
    path = CORPUS / f'{clip_name}.mp4'
    cap = cv2.VideoCapture(str(path))
    frames = []
    # warmup on first frame (JIT, memory alloc)
    ok, f0 = cap.read()
    if not ok:
        raise RuntimeError(f'cannot read {path}')
    for _ in range(warmup):
        adapter.detect(f0)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = time.perf_counter()
        ents = adapter.detect(frame)
        lat = (time.perf_counter() - t0) * 1000
        frames.append({'i': i, 'latency_ms': round(lat, 3), 'entities': ents})
        i += 1
    cap.release()
    out_dir = RESULTS / adapter.name
    out_dir.mkdir(exist_ok=True)
    payload = {'model': adapter.name, 'kind': adapter.kind, 'clip': clip_name,
               'expected_entities': adapter.expected_entities(CLIP_PEOPLE[clip_name]) if CLIP_PEOPLE.get(clip_name) else None,
               'frames': frames}
    (out_dir / f'{clip_name}.json').write_text(json.dumps(payload))
    print(f'  {adapter.name} x {clip_name}: {len(frames)} frames, '
          f'p50={statistics.median(f["latency_ms"] for f in frames):.1f}ms')
    return payload

In [5]:
# RUN ALL (this is the long cell — one model at a time over all clips)
clips = [c for c in CLIP_PEOPLE if (CORPUS / f'{c}.mp4').exists()]
print(f'Running on clips: {clips}')

adapters = [YOLOBodyAdapter, YOLOHandAdapter, MediaPipeAdapter]
if Path(RTMPOSE_ONNX).exists():
    adapters.append(RTMPoseBodyAdapter)

for make in adapters:
    adapter = make()
    print(f'== {adapter.name} ==')
    for clip in clips:
        run_model_on_clip(adapter, clip)
    if hasattr(adapter, 'close'):
        adapter.close()
print('ALL RUNS COMPLETE')

Running on clips: ['idle_1m', 'idle_2m', 'idle_3m', 'slash_1m', 'slash_2m', 'slash_3m', 'two_idle_2m', 'two_slash_2m', 'two_cross_2m', 'walkon_2m']


== yolo26_body ==


  yolo26_body x idle_1m: 452 frames, p50=11.5ms


  yolo26_body x idle_2m: 451 frames, p50=11.5ms


  yolo26_body x idle_3m: 451 frames, p50=11.7ms


...

  rtmpose_body x slash_3m: 600 frames, p50=35.3ms


  rtmpose_body x two_idle_2m: 450 frames, p50=38.9ms


  rtmpose_body x two_slash_2m: 601 frames, p50=44.4ms


  rtmpose_body x two_cross_2m: 900 frames, p50=38.5ms


  rtmpose_body x walkon_2m: 451 frames, p50=41.9ms
ALL RUNS COMPLETE


## 3. Metrics

- **Latency** P50/P95/P99 across all clips.
- **Dropout**: frames with fewer entities than expected (protocol guarantees presence). Also **max burst** (consecutive dropped frames).
- **Jitter**: per-track std-dev of wrist position on idle clips (px @720p), via greedy nearest-neighbor tracks.
- **Track churn** (identity proxy): on `two_cross_2m`, number of track births beyond expected — a fully-automatic proxy for ID instability. Qualitative check: overlay video render (section 5).
- **Acquisition**: frames from first appearance to stable detection on `walkon_2m` (manual review of timeline plot).

In [6]:
def load(model, clip):
    p = RESULTS / model / f'{clip}.json'
    return json.loads(p.read_text()) if p.exists() else None

def latency_stats(payloads):
    lats = sorted(f['latency_ms'] for p in payloads for f in p['frames'])
    if not lats: return {}
    n = len(lats)
    return {'p50': lats[n//2], 'p95': lats[int(n*0.95)], 'p99': lats[int(n*0.99)], 'max': lats[-1]}

def dropout(payload):
    exp = payload['expected_entities']
    if exp is None: return None
    drops = [1 if len(f['entities']) < exp else 0 for f in payload['frames']]
    burst = cur = 0
    for d in drops:
        cur = cur + 1 if d else 0
        burst = max(burst, cur)
    return {'rate': sum(drops)/len(drops), 'max_burst': burst, 'frames': len(drops)}

def greedy_tracks(payload, max_dist=0.15):
    """Greedy NN tracker over entity centroids. Returns tracks + churn count."""
    tracks, births = {}, 0  # id -> last (cx,cy); positions per id
    history = {}
    next_id = 0
    for f in payload['frames']:
        unmatched = list(range(len(f['entities'])))
        assigned = {}
        for tid, (px, py) in list(tracks.items()):
            best, bd = None, max_dist
            for j in unmatched:
                e = f['entities'][j]
                d = math.hypot(e['cx']-px, e['cy']-py)
                if d < bd: best, bd = j, d
            if best is not None:
                assigned[tid] = best
                unmatched.remove(best)
        new_tracks = {}
        for tid, j in assigned.items():
            e = f['entities'][j]
            new_tracks[tid] = (e['cx'], e['cy'])
            history.setdefault(tid, []).append((f['i'], e))
        for j in unmatched:  # birth
            e = f['entities'][j]
            new_tracks[next_id] = (e['cx'], e['cy'])
            history.setdefault(next_id, []).append((f['i'], e))
            next_id += 1
            births += 1
        tracks = new_tracks
    return history, births

def jitter_px(payload, w=1280, h=720):
    """Per-track, per-wrist-slot positional std-dev on idle clips.
    Wrist slots are computed separately (mixing left+right wrists would
    count inter-hand distance as jitter)."""
    hist, _ = greedy_tracks(payload)
    stds = []
    for tid, seq in hist.items():
        if len(seq) < 30: continue
        n_slots = max(len(e['wrists']) for _, e in seq)
        for k in range(n_slots):
            pts = [e['wrists'][k] for _, e in seq if len(e['wrists']) > k]
            if len(pts) < 30: continue
            xs = [p[0]*w for p in pts]; ys = [p[1]*h for p in pts]
            stds.append(math.hypot(statistics.pstdev(xs), statistics.pstdev(ys)))
    return statistics.mean(stds) if stds else None

print('Metric functions defined.')

Metric functions defined.


## 4. Analysis Tables

In [7]:
MODELS = ['yolo26_body', 'yolo26_hand', 'mediapipe_hands', 'rtmpose_body']
rows = []
for m in MODELS:
    payloads = [p for c in CLIP_PEOPLE if (p := load(m, c))]
    if not payloads:
        print(f'{m}: no results yet'); continue
    lat = latency_stats(payloads)
    d2 = dropout(load(m, 'slash_2m')) if load(m, 'slash_2m') else None
    d3 = dropout(load(m, 'slash_3m')) if load(m, 'slash_3m') else None
    two = load(m, 'two_slash_2m')
    d_two = dropout(two) if two else None
    cross = load(m, 'two_cross_2m')
    churn = (greedy_tracks(cross)[1] - cross['expected_entities']) if cross else None
    jit = jitter_px(load(m, 'idle_2m')) if load(m, 'idle_2m') else None
    rows.append({'model': m, **{f'lat_{k}': v for k, v in lat.items()},
                 'drop_slash2m': d2 and round(d2['rate']*100, 1), 'burst2m': d2 and d2['max_burst'],
                 'drop_slash3m': d3 and round(d3['rate']*100, 1),
                 'drop_two2m': d_two and round(d_two['rate']*100, 1),
                 'track_churn': churn, 'jitter_px': jit and round(jit, 1)})

hdr = ['model', 'lat_p50', 'lat_p95', 'lat_p99', 'drop_slash2m', 'burst2m', 'drop_slash3m', 'drop_two2m', 'track_churn', 'jitter_px']
print(' | '.join(f'{h:>13}' for h in hdr))
print('-' * (16 * len(hdr)))
for r in rows:
    print(' | '.join(f'{str(r.get(h, "—")):>13}' for h in hdr))

        model |       lat_p50 |       lat_p95 |       lat_p99 |  drop_slash2m |       burst2m |  drop_slash3m |    drop_two2m |   track_churn |     jitter_px
----------------------------------------------------------------------------------------------------------------------------------------------------------------
  yolo26_body |         11.88 |        21.591 |        31.949 |           0.0 |             0 |           0.0 |           0.0 |            65 |          48.4
  yolo26_hand |        12.127 |        21.546 |        29.073 |          98.7 |           505 |         100.0 |         100.0 |            13 |          None
mediapipe_hands |        29.057 |        43.365 |        47.641 |          69.7 |            40 |          78.5 |          92.8 |           252 |          None
 rtmpose_body |        35.787 |        51.563 |        57.047 |           0.0 |             0 |           0.0 |           0.0 |            65 |          44.5


## 5. Qualitative: overlay video render (for two_cross + blog post)

In [8]:
def render_overlay(model, clip):
    payload = load(model, clip)
    if not payload: return
    hist, _ = greedy_tracks(payload)
    # frame -> [(tid, entity)]
    per_frame = {}
    for tid, seq in hist.items():
        for i, e in seq:
            per_frame.setdefault(i, []).append((tid, e))
    colors = [(0,255,0), (255,120,0), (0,120,255), (255,0,255), (0,255,255), (120,120,255)]
    cap = cv2.VideoCapture(str(CORPUS / f'{clip}.mp4'))
    out = cv2.VideoWriter(str(RESULTS / f'overlay_{model}_{clip}.mp4'),
                          cv2.VideoWriter_fourcc(*'mp4v'), 30, (1280, 720))
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        for tid, e in per_frame.get(i, []):
            c = colors[tid % len(colors)]
            cv2.circle(frame, (int(e['cx']*1280), int(e['cy']*720)), 14, c, 3)
            cv2.putText(frame, f'#{tid}', (int(e['cx']*1280)+16, int(e['cy']*720)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, c, 2)
            for wx, wy in e['wrists']:
                cv2.circle(frame, (int(wx*1280), int(wy*720)), 6, c, -1)
        out.write(frame); i += 1
    cap.release(); out.release()
    print(f'wrote overlay_{model}_{clip}.mp4')

for m in MODELS:
    if load(m, 'two_cross_2m'):
        render_overlay(m, 'two_cross_2m')

wrote overlay_yolo26_body_two_cross_2m.mp4


wrote overlay_yolo26_hand_two_cross_2m.mp4


wrote overlay_mediapipe_hands_two_cross_2m.mp4


wrote overlay_rtmpose_body_two_cross_2m.mp4


## 6. Decision: Gates + Lexicographic Ranking

In [9]:
GATE_P95_MS = 33.0     # G1: 30fps budget
GATE_DROP_PCT = 5.0    # G2: slash dropout @2m

print(f"{'model':<18} {'G1 p95<=33':<12} {'G2 drop<=5%':<12} {'G3 2-player':<12} verdict")
survivors = []
for r in rows:
    g1 = r.get('lat_p95') is not None and r['lat_p95'] <= GATE_P95_MS
    g2 = r.get('drop_slash2m') is not None and r['drop_slash2m'] <= GATE_DROP_PCT
    g3 = r.get('drop_two2m') is not None and r['drop_two2m'] < 50  # detects 2p most of the time
    ok = g1 and g2 and g3
    print(f"{r['model']:<18} {str(g1):<12} {str(g2):<12} {str(g3):<12} {'PASS' if ok else 'FAIL'}")
    if ok: survivors.append(r)

# G4 (deployment path) is a documented judgment call: all contenders have ONNX/mobile paths;
# MediaPipe ships via GDMP today. Note per-model in the writeup.

if survivors:
    ranked = sorted(survivors, key=lambda r: (
        r.get('drop_slash2m') or 99, r.get('lat_p95') or 9999,
        r.get('track_churn') or 999, r.get('jitter_px') or 999,
        r.get('drop_slash3m') or 99))
    print('\nLEXICOGRAPHIC RANKING (dropout -> latency -> churn -> jitter -> range):')
    for i, r in enumerate(ranked, 1):
        print(f"  {i}. {r['model']}")
    print(f"\nWINNER: {ranked[0]['model']}")
else:
    print('\nNo model passed all gates — revisit thresholds or optimize contenders.')

model              G1 p95<=33   G2 drop<=5%  G3 2-player  verdict
yolo26_body        True         True         True         PASS
yolo26_hand        True         False        False        FAIL
mediapipe_hands    False        False        False        FAIL
rtmpose_body       False        True         True         FAIL

LEXICOGRAPHIC RANKING (dropout -> latency -> churn -> jitter -> range):
  1. yolo26_body

WINNER: yolo26_body


## 7. Limitations & Future Work

- **Model latency ≠ end-to-end latency.** We measure inference wall-clock only; camera capture → display adds ~30–60 ms that is backend-independent. Photon-to-photon measurement is future work.
- **No spatial ground truth.** Dropout uses protocol-guaranteed presence, not annotation. Keypoint accuracy relies on each model's published dataset metrics.
- **Track churn is a proxy** for identity swaps; the overlay videos are the qualitative check.
- **One hardware target** (RTX 3070 Ti laptop). Phone (Android/iOS) benchmarking is round 2 — the decision may differ on mobile NPUs.
- **Two subjects, one room.** More subjects/lighting conditions would strengthen publishability (workshop-paper tier).
- **RTMPose adapter pending** Docker ONNX export (`models/rtmpose/`).

### Next steps after a winner is declared
1. Integrate winner into Godot (`AIInput` backend swap)
2. Calibration screen (upper-body check + raise-hands-to-start)
3. Blog post from this notebook + publish corpus & harness
4. Round 2 on phone hardware